In [0]:
# ===========================================
# Notebook Name:
# 02_ingest_decks
#
# Purpose:
# Fetch Pokemon deck HTML pages for deck_codes
# selected by the Ops fetch-target table and
# store the raw responses in Bronze.
#
# Input:
# pokemon.ops.deck_fetch_targets
# (joined with pokemon.silver.tournament_results
# for deck_print_url, since deck_fetch_targets
# tracks *what* to fetch, not *how*)
#
# Outputs:
# pokemon.bronze.deck_raw
# pokemon.bronze.scrape_log (source_type='deck')
#
# Data quality:
# - deck_code must not be null
# - HTTP response must be successful
# - HTML must contain the deck code
# - identical responses are not duplicated
# ===========================================

In [0]:
from datetime import datetime, timezone
from typing import Any
import hashlib
import time
import uuid

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

DECK_FETCH_TARGETS_TABLE = (
    "pokemon.ops.deck_fetch_targets"
)

TOURNAMENT_RESULTS_TABLE = (
    "pokemon.silver.tournament_results"
)

DECK_RAW_TABLE = (
    "pokemon.bronze.deck_raw"
)

SCRAPE_LOG_TABLE = (
    "pokemon.bronze.scrape_log"
)

REQUEST_INTERVAL_SECONDS = 1.5
REQUEST_TIMEOUT_SECONDS = 30

MAX_FETCH_TARGETS = 500

print("Input :", DECK_FETCH_TARGETS_TABLE)
print("Output:", DECK_RAW_TABLE)
print("Log   :", SCRAPE_LOG_TABLE)

In [ ]:
# -------------------------------------------
# One pipeline_run_id per notebook execution,
# shared by every scrape_log row this run writes,
# so a single Workflow run can be traced across
# tasks/notebooks. Distinct from the per-request
# request_id assigned to each deck fetch below.
# Falls back to a fresh uuid when run
# interactively outside a Databricks Job (no
# upstream initialize_pipeline_run task to read
# from).
# -------------------------------------------
pipeline_run_id = dbutils.jobs.taskValues.get(
    taskKey="initialize_pipeline_run",
    key="pipeline_run_id",
    default=None,
    debugValue="manual-run",
)

PIPELINE_RUN_ID = (
    pipeline_run_id
    if pipeline_run_id
    else str(uuid.uuid4())
)

print("pipeline_run_id:", PIPELINE_RUN_ID)

In [0]:
# deck_fetch_targets records *which* deck_codes
# should be (re)fetched and why, but does not carry
# the print URL used to fetch the HTML. That URL
# lives on tournament_results (one row per deck_code
# result appearance), so join it in here.
fetch_targets_df = (
    spark.table(DECK_FETCH_TARGETS_TABLE)
    .select(
        "deck_code",
        "fetch_reason",
        "priority",
    )
    .filter(
        F.col("deck_code").isNotNull()
    )
)

deck_print_url_by_code_df = (
    spark.table(TOURNAMENT_RESULTS_TABLE)
    .select(
        "deck_code",
        "deck_print_url",
    )
    .filter(F.col("deck_code").isNotNull())
    .filter(F.col("deck_print_url").isNotNull())
    .dropDuplicates(["deck_code"])
)

pending_decks_df = (
    fetch_targets_df.alias("target")
    .join(
        deck_print_url_by_code_df.alias("url"),
        on="deck_code",
        how="inner",
    )
    .select(
        "deck_code",
        "deck_print_url",
        "fetch_reason",
        "priority",
    )
    .orderBy(
        F.col("priority").asc_nulls_last(),
        F.col("deck_code").desc(),
    )
    .limit(
        MAX_FETCH_TARGETS
    )
)

pending_count = pending_decks_df.count()

display(
    pending_decks_df
)

print("Pending deck count:", pending_count)

missing_url_count = (
    fetch_targets_df.count() - pending_count
)

if missing_url_count > 0:
    print(
        f"Skipped {missing_url_count} fetch "
        "target(s) with no known deck_print_url "
        "in tournament_results."
    )

In [0]:
pending_decks = [
    row.asDict()
    for row in pending_decks_df.collect()
]

print("Decks to fetch:", len(pending_decks))

for deck in pending_decks[:5]:
    print(deck)

In [0]:
REQUEST_HEADERS = {
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,*/*;q=0.8"
    ),
    "Accept-Language": "ja,en-US;q=0.9,en;q=0.8",
    "User-Agent": (
        "Mozilla/5.0 "
        "(compatible; PokemonLakehouse/0.1)"
    ),
}


def fetch_deck_html(
    deck_code: str,
    deck_print_url: str,
    timeout: int = REQUEST_TIMEOUT_SECONDS,
) -> dict[str, Any]:

    started_at = time.perf_counter()
    scraped_at = datetime.now(timezone.utc)

    response = requests.get(
        deck_print_url,
        headers=REQUEST_HEADERS,
        timeout=timeout,
    )

    elapsed_ms = int(
        (time.perf_counter() - started_at) * 1000
    )

    response.raise_for_status()

    response.encoding = (
        response.apparent_encoding
        or response.encoding
        or "utf-8"
    )

    html = response.text

    if not html.strip():
        raise ValueError(
            f"Empty HTML returned for deck {deck_code}"
        )

    if deck_code not in html:
        raise ValueError(
            f"Deck code was not found in HTML: {deck_code}"
        )

    response_hash = hashlib.sha256(
        html.encode("utf-8")
    ).hexdigest()

    return {
        "deck_code": deck_code,
        "source_url": response.url,
        "http_status": response.status_code,
        "payload": html,
        "payload_format": "html",
        "response_hash": response_hash,
        "scraped_at": scraped_at,
        "elapsed_ms": elapsed_ms,
    }

In [0]:
if pending_decks:
    test_deck = pending_decks[0]

    test_result = fetch_deck_html(
        deck_code=test_deck["deck_code"],
        deck_print_url=test_deck["deck_print_url"],
    )

    print("deck_code:", test_result["deck_code"])
    print("status:", test_result["http_status"])
    print("HTML length:", len(test_result["payload"]))
    print("hash:", test_result["response_hash"])
else:
    print("No pending decks.")

In [ ]:
raw_rows: list[dict[str, Any]] = []
log_rows: list[dict[str, Any]] = []

total = len(pending_decks)

for index, deck in enumerate(
    pending_decks,
    start=1,
):
    deck_code = deck["deck_code"]
    deck_print_url = deck["deck_print_url"]
    fetch_reason = deck["fetch_reason"] or "unknown"
    request_id = str(uuid.uuid4())
    execution_time = datetime.now(timezone.utc)

    print(
        f"[{index}/{total}] Fetching {deck_code} "
        f"(reason={fetch_reason})"
    )

    try:
        result = fetch_deck_html(
            deck_code=deck_code,
            deck_print_url=deck_print_url,
        )

        raw_rows.append({
            "deck_code": result["deck_code"],
            "source_url": result["source_url"],
            "http_status": result["http_status"],
            "payload": result["payload"],
            "payload_format": result["payload_format"],
            "response_hash": result["response_hash"],
            "scraped_at": result["scraped_at"],
            "ingestion_date": result["scraped_at"].date(),
        })

        log_rows.append({
            "request_id": request_id,
            "pipeline_run_id": PIPELINE_RUN_ID,
            "source_type": "deck",
            "source_id": deck_code,
            "request_url": result["source_url"],
            "http_status": result["http_status"],
            "status": "success",
            "elapsed_ms": result["elapsed_ms"],
            "records_found": 1,
            "error_type": None,
            "error_message": None,
            "scraped_at": result["scraped_at"],
            "ingestion_date": result["scraped_at"].date(),
        })

        print(
            f"  Success: {len(result['payload'])} bytes"
        )

    except Exception as error:
        error_message = str(error)[:2000]

        log_rows.append({
            "request_id": request_id,
            "pipeline_run_id": PIPELINE_RUN_ID,
            "source_type": "deck",
            "source_id": deck_code,
            "request_url": deck_print_url,
            "http_status": None,
            "status": "error",
            "elapsed_ms": None,
            "records_found": 0,
            "error_type": type(error).__name__,
            "error_message": error_message,
            "scraped_at": execution_time,
            "ingestion_date": execution_time.date(),
        })

        print(f"  Failed: {error}")

    if index < total:
        time.sleep(REQUEST_INTERVAL_SECONDS)

print("Success count:", len(raw_rows))
print(
    "Failure count:",
    sum(row["status"] == "error" for row in log_rows),
)

In [ ]:
deck_raw_schema = StructType([
    StructField("deck_code", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("http_status", IntegerType(), True),
    StructField("payload", StringType(), False),
    StructField("payload_format", StringType(), False),
    StructField("response_hash", StringType(), False),
    StructField("scraped_at", TimestampType(), False),
    StructField("ingestion_date", DateType(), False),
])

scrape_log_schema = StructType([
    StructField("request_id", StringType(), False),
    StructField("pipeline_run_id", StringType(), False),
    StructField("source_type", StringType(), False),
    StructField("source_id", StringType(), True),
    StructField("request_url", StringType(), True),
    StructField("http_status", IntegerType(), True),
    StructField("status", StringType(), False),
    StructField("elapsed_ms", LongType(), True),
    StructField("records_found", IntegerType(), True),
    StructField("error_type", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("scraped_at", TimestampType(), False),
    StructField("ingestion_date", DateType(), False),
])

In [ ]:
if raw_rows:
    deck_raw_df = (
        spark.createDataFrame(
            raw_rows,
            schema=deck_raw_schema,
        )
    )

    # deck_fetch_targets normally excludes deck_codes
    # that already have a raw row (deck content is
    # fetch-once), but manual_refetch and retry_error
    # targets can still re-request a deck_code that
    # already succeeded earlier. Since deck HTML for a
    # given deck_code is expected to be byte-identical
    # across fetches, anti-join against the existing
    # table on (deck_code, response_hash) so an
    # unchanged re-fetch is not appended as a duplicate
    # version -- that duplicate would otherwise trip the
    # validation in the cell below.
    new_deck_raw_df = (
        deck_raw_df.alias("new")
        .join(
            spark.table(DECK_RAW_TABLE)
            .select(
                "deck_code",
                "response_hash",
            )
            .alias("existing"),
            on=[
                "deck_code",
                "response_hash",
            ],
            how="left_anti",
        )
    )

    inserted_count = new_deck_raw_df.count()

    display(
        new_deck_raw_df.select(
            "deck_code",
            "http_status",
            F.length("payload").alias("payload_length"),
            "response_hash",
            "scraped_at",
        )
    )

    (
        new_deck_raw_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(DECK_RAW_TABLE)
    )

    print(
        f"{inserted_count} rows inserted "
        f"into {DECK_RAW_TABLE}"
    )

    skipped_unchanged_count = (
        len(raw_rows) - inserted_count
    )

    if skipped_unchanged_count > 0:
        print(
            f"  {skipped_unchanged_count} row(s) "
            "matched an existing (deck_code, "
            "response_hash) and were not "
            "re-inserted."
        )
else:
    print("No successful deck responses.")

In [0]:
if log_rows:
    scrape_log_df = (
        spark.createDataFrame(
            log_rows,
            schema=scrape_log_schema,
        )
    )

    (
        scrape_log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(SCRAPE_LOG_TABLE)
    )

    display(scrape_log_df)

    print(
        f"{len(log_rows)} rows inserted "
        f"into {SCRAPE_LOG_TABLE}"
    )
else:
    print("No scrape logs to write.")

In [0]:
fetched_deck_codes = [
    row["deck_code"] for row in raw_rows
]

if fetched_deck_codes:
    display(
        spark.table(DECK_RAW_TABLE)
        .filter(
            F.col("deck_code").isin(
                fetched_deck_codes
            )
        )
        .select(
            "deck_code",
            "http_status",
            "payload_format",
            F.length("payload").alias("payload_length"),
            "response_hash",
            "scraped_at",
        )
        .orderBy(F.col("scraped_at").desc())
    )
else:
    print("No deck_raw rows fetched this run.")

In [0]:
display(
    spark.table(DECK_RAW_TABLE)
    .agg(
        F.count("*").alias("raw_response_count"),
        F.countDistinct("deck_code").alias(
            "unique_deck_count"
        ),
        F.min(F.length("payload")).alias(
            "minimum_payload_length"
        ),
        F.max(F.length("payload")).alias(
            "maximum_payload_length"
        ),
    )
)

In [0]:
display(
    spark.table(SCRAPE_LOG_TABLE)
    .filter(F.col("source_type") == "deck")
    .orderBy(F.col("scraped_at").desc())
    .limit(20)
)

In [0]:
duplicate_versions_df = (
    spark.table(DECK_RAW_TABLE)
    .groupBy(
        "deck_code",
        "response_hash",
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_versions_df.count()

if duplicate_count > 0:
    display(duplicate_versions_df)

    raise ValueError(
        f"{duplicate_count} duplicate deck "
        "response versions detected"
    )

print(
    "Validation passed: "
    "no duplicate deck response versions"
)

In [0]:
error_count = sum(
    row["status"] == "error" for row in log_rows
)

if error_count > 0:
    raise RuntimeError(
        "Some deck HTML requests failed. "
        f"error_count={error_count}"
    )